In [1]:
!pip install fastapi uvicorn ultralytics opencv-python-headless sqlalchemy python-multipart pyngrok nest-asyncio -q
print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.4 MB/s eta 0:00:00
✅ All packages installed


In [2]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite:///./roadwatch.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

class Detection(Base):
    __tablename__ = "detections"
    id         = Column(Integer, primary_key=True, index=True)
    location   = Column(String)
    damage     = Column(String)
    severity   = Column(String)
    confidence = Column(Float)
    frame      = Column(Integer)
    timestamp  = Column(String)

Base.metadata.create_all(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

print("✅ Database ready")

✅ Database ready


/tmp/ipykernel_7327/2488069118.py:8: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
!rm -rf SafeRaasta
!git clone https://github.com/saiyam-bajpai/SafeRaasta.git
!cp SafeRaasta/outputs/best.pt best.pt

import os
if os.path.exists("best.pt"):
    print("✅ best.pt ready!")
else:
    print("❌ Error — best.pt nahi mila")

Cloning into 'SafeRaasta'...
remote: Enumerating objects: 13297, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 13297 (delta 0), reused 0 (delta 0), pack-reused 13296 (from 2)
Receiving objects: 100% (13297/13297), 488.01 MiB | 25.19 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Updating files: 100% (13309/13309), done.
✅ best.pt ready!


In [4]:
import cv2
from ultralytics import YOLO

model = YOLO("best.pt")
print("✅ AI Model loaded!")
print("Classes:", model.names)

def calculate_severity(area):
    if area < 1000:
        return "low"
    elif area < 3000:
        return "medium"
    else:
        return "high"

def run_detection_on_frame(frame, frame_number, location="Unknown Road"):
    detections = []
    results = model(frame, verbose=False)
    for box in results[0].boxes:
        conf = float(box.conf[0])
        if conf < 0.5:
            continue
        cls = int(box.cls[0])
        x1, y1, x2, y2 = box.xyxy[0]
        area = (x2 - x1) * (y2 - y1)
        detection = {
            "location":   location,
            "damage":     results[0].names[cls],
            "severity":   calculate_severity(float(area)),
            "confidence": round(conf, 2),
            "frame":      frame_number,
            "timestamp":  f"00:{frame_number // 1800:02d}:{(frame_number // 30) % 60:02d}"
        }
        detections.append(detection)
    return detections

def process_video(video_path: str, location: str = "Unknown Road"):
    cap = cv2.VideoCapture(video_path)
    all_detections = []
    frame_number = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        if frame_number % 10 == 0:
            detections = run_detection_on_frame(frame, frame_number, location)
            all_detections.extend(detections)
        frame_number += 1
    cap.release()
    return all_detections

print("✅ Real AI Detector ready!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ AI Model loaded!
Classes: {0: 'Pothole'}
✅ Real AI Detector ready!


In [5]:
from fastapi import FastAPI, UploadFile, File, Depends, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session
import shutil, os

app = FastAPI(title="RoadWatch AI Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def get_db_direct():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.get("/")
def root():
    return {"message": "RoadWatch AI Backend is running ✅"}

@app.post("/api/upload")
async def upload_video(
    background_tasks: BackgroundTasks,
    location: str = "Unknown Road",
    file: UploadFile = File(...),
    db: Session = Depends(get_db_direct)
):
    os.makedirs("uploads", exist_ok=True)
    save_path = f"uploads/{file.filename}"
    with open(save_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    background_tasks.add_task(process_and_save, save_path, location)
    return {"message": "Video uploaded. Processing started.", "file": file.filename}

def process_and_save(video_path, location):
    db = SessionLocal()
    detections = process_video(video_path, location)
    for d in detections:
        record = Detection(**d)
        db.add(record)
    db.commit()
    db.close()
    print(f"✅ Processing done! {len(detections)} detections saved.")

@app.get("/api/detections")
def get_detections(db: Session = Depends(get_db_direct)):
    results = db.query(Detection).all()
    return [{"id": d.id, "location": d.location, "damage": d.damage,
             "severity": d.severity, "confidence": d.confidence,
             "frame": d.frame, "timestamp": d.timestamp} for d in results]

@app.get("/api/analytics")
def get_analytics(db: Session = Depends(get_db_direct)):
    all_d = db.query(Detection).all()
    return {
        "total_detections": len(all_d),
        "high_severity":    sum(1 for d in all_d if d.severity == "high"),
        "medium_severity":  sum(1 for d in all_d if d.severity == "medium"),
        "low_severity":     sum(1 for d in all_d if d.severity == "low"),
        "damage_types": {
            "pothole":      sum(1 for d in all_d if d.damage == "pothole"),
            "crack":        sum(1 for d in all_d if d.damage == "crack"),
            "damaged road": sum(1 for d in all_d if d.damage == "damaged road"),
        }
    }

@app.get("/api/live-status")
def live_status(db: Session = Depends(get_db_direct)):
    count = db.query(Detection).count()
    return {"status": "active", "total_detections_so_far": count}

print("✅ Full FastAPI app ready")

✅ Full FastAPI app ready


In [8]:
!ngrok config add-authtoken 3EO5UtUZjjOUYdQwYOIa18m7t6R_39hwptByGquqJimFs8Swo
print("✅ Auth token saved!")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ Auth token saved!


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.kill()

public_url = ngrok.connect(8000)
print("🌍 Public API URL:", public_url.public_url)
print("📖 API Docs:", public_url.public_url + "/docs")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

🌍 Public API URL: https://clergyman-semester-discolor.ngrok-free.dev
📖 API Docs: https://clergyman-semester-discolor.ngrok-free.dev/docs


INFO:     Started server process [7327]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "POST /api/upload?location=Bhopal%20Road HTTP/1.1" 200 OK
✅ Processing done! 83 detections saved.
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /api/detections HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /api/detections HTTP/1.1" 200 OK
INFO:     2409:40c4:ec:1399:ed64:18f9:a643:ae7b:0 - "GET /api/analytics HTTP/1.1" 200 OK
